# Test Optimized WGAN Package

This notebook tests the optimized WGAN package and compares results with the original tutorial.

In [ ]:
# Install the optimized package (run once)
import sys
sys.path.insert(0, '/Users/anzony.quisperojas/Documents/GitHub/python/dswganoptx')

import pandas as pd
import wgan
import torch
from time import time

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Load Data

Load the CPS dataset from the original repository.

In [ ]:
# Load data from local file
df = pd.read_feather('/Users/anzony.quisperojas/Documents/GitHub/python/ds-wgan/data/original_data/cps.feather')
df = df.drop(["u74", "u75"], axis=1)
print(f"Dataset shape: {df.shape}")
df.head()

## Setup Training

Create balanced dataset and initialize models (same as tutorial).

In [ ]:
# Create balanced dataset
df_balanced = df.sample(2*len(df), weights=(1-df.t.mean())*df.t+df.t.mean()*(1-df.t), replace=True)

# X | t
continuous_vars_0 = ["age", "education", "re74", "re75"]
continuous_lower_bounds_0 = {"re74": 0, "re75": 0}
categorical_vars_0 = ["black", "hispanic", "married", "nodegree"]
context_vars_0 = ["t"]

# Y | X, t
continuous_vars_1 = ["re78"]
continuous_lower_bounds_1 = {"re78": 0}
categorical_vars_1 = []
context_vars_1 = ["t", "age", "education", "re74", "re75", "black", "hispanic", "married", "nodegree"]

# Use CPU if CUDA not available, otherwise use CUDA
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Initialize objects (same parameters as tutorial)
data_wrappers = [
    wgan.DataWrapper(df_balanced, continuous_vars_0, categorical_vars_0,
                     context_vars_0, continuous_lower_bounds_0),
    wgan.DataWrapper(df_balanced, continuous_vars_1, categorical_vars_1,
                     context_vars_1, continuous_lower_bounds_1)
]

specs = [
    wgan.Specifications(dw, batch_size=4096, max_epochs=1000, critic_lr=1e-3, 
                        generator_lr=1e-3, print_every=100, device=device)
    for dw in data_wrappers
]

generators = [wgan.Generator(spec) for spec in specs]
critics = [wgan.Critic(spec) for spec in specs]

## Train Model 0: X | t

Train the first model (covariates conditional on treatment).

In [ ]:
# Train X | t
print("Training Model 0: X | t")
print("=" * 50)

start_time = time()
x, context = data_wrappers[0].preprocess(df_balanced)
wgan.train(generators[0], critics[0], x, context, specs[0])
time_model0 = time() - start_time

print(f"\nTotal training time for Model 0: {time_model0:.1f} seconds")

## Train Model 1: Y | X, t

Train the second model (outcome conditional on covariates and treatment).

In [ ]:
# Train Y | X, t
print("Training Model 1: Y | X, t")
print("=" * 50)

start_time = time()
x, context = data_wrappers[1].preprocess(df_balanced)
wgan.train(generators[1], critics[1], x, context, specs[1])
time_model1 = time() - start_time

print(f"\nTotal training time for Model 1: {time_model1:.1f} seconds")

## Generate Data and Compute ATT

Generate synthetic data and compute the Average Treatment Effect on the Treated (ATT).

In [ ]:
# Generate data
print("Generating synthetic data...")
start_time = time()

df_generated = data_wrappers[0].apply_generator(generators[0], df.sample(int(1e6), replace=True))
df_generated = data_wrappers[1].apply_generator(generators[1], df_generated)

# Add counterfactual outcomes
from copy import copy
df_generated_cf = copy(df_generated)
df_generated_cf["t"] = 1 - df_generated_cf["t"]
df_generated["re78_cf"] = data_wrappers[1].apply_generator(generators[1], df_generated_cf)["re78"]

generation_time = time() - start_time
print(f"Generation time: {generation_time:.1f} seconds")

In [ ]:
# Compute ATT
att = ((df_generated.re78 - df_generated.re78_cf) * (2*df_generated.t - 1))[df_generated.t == 1].mean()
print(f"ATT: {att:.2f}")

# This should be similar to the original tutorial results

## Compare Distributions

Visualize the comparison between real and generated data.

In [ ]:
# Compare distributions
wgan.compare_dfs(
    df, df_generated,
    scatterplot=dict(x=["re74", "age", "education"], y=["re78", "re75"], samples=400, smooth=0),
    table_groupby=["t"],
    histogram=dict(variables=["re78", "re74", "age", "education"], nrow=2, ncol=2),
    figsize=3
)

## Performance Summary

In [ ]:
print("\n" + "=" * 50)
print("PERFORMANCE SUMMARY")
print("=" * 50)
print(f"Device: {device}")
print(f"Model 0 (X|t) training time: {time_model0:.1f} seconds")
print(f"Model 1 (Y|X,t) training time: {time_model1:.1f} seconds")
print(f"Total training time: {time_model0 + time_model1:.1f} seconds")
print(f"Data generation time (1M samples): {generation_time:.1f} seconds")
print(f"\nATT estimate: {att:.2f}")
print("\nNote: Original tutorial showed ~31 sec per 100 epochs on GPU")